# 1 Extracting Data

### 1.0 Imports

In [ ]:
import pandas as pd
from fitparse import FitFile
from collections import Counter
from pathlib import Path
from datetime import datetime, timedelta

In [ ]:
# ============================================================
# Parameters
# ============================================================
first_date =  "2026-06-03"

LOCAL_OFFSET = pd.Timedelta(hours=2)

GARMIN_EPOCH = datetime(1989, 12, 31)

In [ ]:
# ============================================================
# Select date and inspect WELLNESS files
# ============================================================
data_dir = Path(f"../data/{first_date}")
wellness_files = sorted(data_dir.glob("*_WELLNESS.fit"))

print(f"Found {len(wellness_files)} WELLNESS files\n")

summary = []

for file in wellness_files:
    fitfile = FitFile(str(file))

    stress_count = 0

    for _ in fitfile.get_messages("stress_level"):
        stress_count += 1

    summary.append({
        "file": file.name,
        "stress_records": stress_count
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

### 1.1 Stress

In [ ]:
def extract_stress(
    wellness_files,
    curr_date,
    local_offset
):
    """
    Extract stress data from all WELLNESS files for one day.
    """

    all_rows = []

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("stress_level"):

            row = msg.get_values()

            all_rows.append({
                "source_file": file.name,
                "timestamp": row["stress_level_time"] + local_offset,
                "stress": row["stress_level_value"],
                "body_battery": row.get("body_battery")
            })

    stress_df = pd.DataFrame(all_rows)

    # Garmin missing value
    stress_df["stress"] = (
        stress_df["stress"]
        .replace(-1, pd.NA)
    )

    stress_df = (
        stress_df
        .sort_values("timestamp")
        .drop_duplicates(
            subset="timestamp",
            keep="last"
        )
        .reset_index(drop=True)
    )

    stress_df = stress_df[
        stress_df["timestamp"].dt.date
        == pd.to_datetime(curr_date).date()
    ].copy()

    return stress_df

In [ ]:
stress_df = extract_stress(
    wellness_files,
    first_date,
    LOCAL_OFFSET
)

print("Rows:", len(stress_df))
print("Start:", stress_df["timestamp"].min())
print("End:", stress_df["timestamp"].max())

display(stress_df.head())
display(stress_df.tail())

### 1.2 Monitor Data

In [ ]:
def extract_monitoring(
    wellness_files,
    curr_date
):
    """
    Extract monitoring data from all WELLNESS files for one day.
    Includes heart rate and activity-related monitoring fields.
    """

    rows = []

    target_date = pd.to_datetime(curr_date).date()

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("monitoring"):

            row = msg.get_values()

            if "timestamp_16" not in row:
                continue

            timestamp = (
                datetime.combine(
                    target_date,
                    datetime.min.time()
                )
                + timedelta(seconds=row["timestamp_16"])
            )

            rows.append({
                "source_file": file.name,
                "timestamp": timestamp,

                "heart_rate": row.get("heart_rate"),

                "activity_type": row.get("activity_type"),
                "intensity": row.get("intensity"),

                "cycles": row.get("cycles"),
                "active_calories": row.get("active_calories"),
                "active_time": row.get("active_time")
            })

    monitoring_df = (
        pd.DataFrame(rows)
        .sort_values("timestamp")
        .drop_duplicates(subset="timestamp", keep="last")
        .reset_index(drop=True)
    )

    return monitoring_df

In [ ]:
monitoring_df = extract_monitoring(
    wellness_files,
    first_date
)

print("Rows:", len(monitoring_df))
print("Start:", monitoring_df["timestamp"].min())
print("End:", monitoring_df["timestamp"].max())

display(monitoring_df.head())
display(monitoring_df.tail())

In [ ]:
monitoring_df.notna().sum().sort_values(ascending=False)

### 1.3 Body Battery

In [ ]:
def extract_body_battery(
    wellness_files,
    curr_date,
    local_offset
):
    """
    Extract body battery data from all WELLNESS files for one day.
    """

    all_rows = []

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("stress_level"):

            row = msg.get_values()

            all_rows.append({
                "source_file": file.name,
                "timestamp": row["stress_level_time"] + local_offset,
                "body_battery": row["unknown_3"]
            })

    body_battery_df = pd.DataFrame(all_rows)

    body_battery_df["body_battery"] = (
        body_battery_df["body_battery"]
        .replace(-1, pd.NA)
    )

    body_battery_df = (
        body_battery_df
        .sort_values("timestamp")
        .drop_duplicates(
            subset="timestamp",
            keep="last"
        )
        .reset_index(drop=True)
    )

    body_battery_df = body_battery_df[
        body_battery_df["timestamp"].dt.date
        == pd.to_datetime(curr_date).date()
    ].copy()

    return body_battery_df

In [ ]:
body_battery_df = extract_body_battery(
    wellness_files,
    first_date,
    LOCAL_OFFSET
)

print("Rows:", len(body_battery_df))
print("Start:", body_battery_df["timestamp"].min())
print("End:", body_battery_df["timestamp"].max())

display(body_battery_df.head())
display(body_battery_df.tail())

### Respirational Rate

In [ ]:
def extract_respiration(
    wellness_files,
    curr_date,
    local_offset,
    garmin_epoch
):
    """
    Extract respiration data from all WELLNESS files for one day.
    """

    all_rows = []

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("unknown_297"):

            row = msg.get_values()

            all_rows.append({
                "source_file": file.name,
                "timestamp": (
                    garmin_epoch
                    + timedelta(seconds=row["unknown_253"])
                    + local_offset
                ),
                "respiration_rate": row["unknown_0"] / 100
            })

    respiration_df = pd.DataFrame(all_rows)

    respiration_df.loc[
        respiration_df["respiration_rate"] < 0,
        "respiration_rate"
    ] = pd.NA

    respiration_df = (
        respiration_df
        .sort_values("timestamp")
        .drop_duplicates(
            subset="timestamp",
            keep="last"
        )
        .reset_index(drop=True)
    )

    respiration_df = respiration_df[
        respiration_df["timestamp"].dt.date
        == pd.to_datetime(curr_date).date()
    ].copy()

    return respiration_df

In [ ]:
respiration_df = extract_respiration(
    wellness_files,
    first_date,
    LOCAL_OFFSET,
    GARMIN_EPOCH
)

print("Rows:", len(respiration_df))
print("Start:", respiration_df["timestamp"].min())
print("End:", respiration_df["timestamp"].max())

display(respiration_df.head())
display(respiration_df.tail())

### 1.5 Merging

In [ ]:
def build_daily_df(
    stress_df,
    body_battery_df,
    respiration_df,
    monitoring_df,
    curr_date
):
    """
    Merge all extracted Garmin data into a unified
    1-minute dataframe for one day.
    """

    # ========================================================
    # Ensure datetime dtype
    # ========================================================

    for df in [
        stress_df,
        body_battery_df,
        respiration_df,
        monitoring_df
    ]:
        df["timestamp"] = pd.to_datetime(df["timestamp"])

    # ========================================================
    # Keep only target day
    # ========================================================

    target_date = pd.to_datetime(curr_date).date()

    stress_df = stress_df[
        stress_df["timestamp"].dt.date == target_date
    ].copy()

    body_battery_df = body_battery_df[
        body_battery_df["timestamp"].dt.date == target_date
    ].copy()

    respiration_df = respiration_df[
        respiration_df["timestamp"].dt.date == target_date
    ].copy()

    monitoring_df = monitoring_df[
        monitoring_df["timestamp"].dt.date == target_date
    ].copy()

    # ========================================================
    # Full minute timeline
    # ========================================================

    full_index = pd.date_range(
        start=pd.Timestamp(curr_date),
        periods=1440,
        freq="1min"
    )

    # ========================================================
    # Helper
    # ========================================================

    def prepare_minute_df(df, value_col):

        temp = df.copy()

        temp["timestamp"] = temp["timestamp"].dt.floor("min")

        return (
            temp.groupby("timestamp")[value_col]
            .first()
            .to_frame()
        )

    # ========================================================
    # Build minute-level tables
    # ========================================================

    stress_min = prepare_minute_df(
        stress_df,
        "stress"
    )

    body_battery_min = prepare_minute_df(
        body_battery_df,
        "body_battery"
    )

    respiration_min = prepare_minute_df(
        respiration_df,
        "respiration_rate"
    )

    heart_rate_min = prepare_minute_df(
        monitoring_df,
        "heart_rate"
    )

    activity_type_min = prepare_minute_df(
        monitoring_df,
        "activity_type"
    )

    intensity_min = prepare_minute_df(
        monitoring_df,
        "intensity"
    )

    cycles_min = prepare_minute_df(
        monitoring_df,
        "cycles"
    )

    active_time_min = prepare_minute_df(
        monitoring_df,
        "active_time"
    )

    active_calories_min = prepare_minute_df(
        monitoring_df,
        "active_calories"
    )

    # ========================================================
    # Merge
    # ========================================================

    daily_df = pd.DataFrame(index=full_index)

    for feature_df in [
        stress_min,
        body_battery_min,
        respiration_min,
        heart_rate_min,
        activity_type_min,
        intensity_min,
        cycles_min,
        active_time_min,
        active_calories_min
    ]:
        daily_df = daily_df.join(feature_df)

    daily_df.index.name = "timestamp"

    return daily_df.reset_index()

In [ ]:
daily_df = build_daily_df(
    stress_df,
    body_battery_df,
    respiration_df,
    monitoring_df,
    first_date
)

print("Shape:", daily_df.shape)

print("\nMissing values:")
print(daily_df.isna().sum())

print("\nDate range:")
print(daily_df["timestamp"].min())
print(daily_df["timestamp"].max())

display(daily_df.head(20))

### 1.6 Save Function

In [ ]:
def save_daily_df(
    daily_df,
    curr_date,
    output_dir="../final_dfs"
):
    """
    Save one extracted day as a CSV file.
    """

    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_file = output_dir / f"{curr_date}.csv"

    daily_df.to_csv(
        output_file,
        index=False
    )

    print(f"Saved: {output_file}")

### 1.7 Data Extraction

In [ ]:
def extract_day(
    curr_date,
    local_offset,
    garmin_epoch
):
    """
    Complete extraction pipeline for one day.
    """

    data_dir = Path(f"../data/{curr_date}")

    wellness_files = sorted(
        data_dir.glob("*_WELLNESS.fit")
    )

    stress_df = extract_stress(
        wellness_files,
        curr_date,
        local_offset
    )

    body_battery_df = extract_body_battery(
        wellness_files,
        curr_date,
        local_offset
    )

    respiration_df = extract_respiration(
        wellness_files,
        curr_date,
        local_offset,
        garmin_epoch
    )

    monitoring_df = extract_monitoring(
        wellness_files,
        curr_date
    )

    daily_df = build_daily_df(
        stress_df,
        body_battery_df,
        respiration_df,
        monitoring_df,
        curr_date
    )

    return daily_df

### 1.8 Extracting multiple days at once

In [ ]:
# ============================================================
# Process all dates
# ============================================================

dates = [
    "2026-06-03"
]

for curr_date in dates:

    print(f"\nProcessing {curr_date}")

    daily_df = extract_day(
        curr_date,
        LOCAL_OFFSET,
        GARMIN_EPOCH
    )

    save_daily_df(
        daily_df,
        curr_date
    )

    print("Shape:", daily_df.shape)

    print("\nMissing values:")
    print(daily_df.isna().sum())

    display(daily_df.head())

# 2 EDA

### Missing Values

In [ ]:
daily_df.isna().sum()

# 3 Preprocessing

In [ ]:
daily_df_processed = daily_df.copy()

### 3.1 Missing Values

In [ ]:
# Fill activity features with forward fill
activity_cols = [
    "activity_type",
    "intensity",
    "cycles",
    "active_time",
    "active_calories"
]

daily_df_processed[activity_cols] = (
    daily_df_processed[activity_cols]
    .ffill()
)

In [ ]:
daily_df_processed.isna().sum()